# Data Wrangling and Merging: IGN Ratings and Steam Engagement Data

## Table of Contents
1. [Data Acquisition](#data-acquisition)
2. [Data Loading and Exploration](#data-loading-and-exploration)
3. [Data Cleaning and Preparation](#data-cleaning-and-preparation)
4. [Data Merging](#data-merging)

## Data Acquisition

We'll download two datasets from Kaggle:
1. **IGN Games Dataset** - Contains professional critics' ratings from IGN
2. **Steam Games Dataset** - Contains user engagement metrics from Steam

In [1]:
# Import necessary libraries
import kagglehub
import os

def print_directory_contents(path):
    """Prints the contents of the given directory."""
    try:
        for item in os.listdir(path):
            print(f"   {item}")
    except FileNotFoundError:
        print(f"The directory {path} does not exist.")

/Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download IGN Games Dataset (Critics' ratings)
ign_dataset_path = kagglehub.dataset_download("joebeachcapital/ign-games")
print("Downloaded IGN Games Dataset (Critics' Ratings).")

# print("Contents:")
# print_directory_contents(ign_dataset_path)

Downloaded IGN Games Dataset (Critics' Ratings).


In [3]:
# Download Steam Games Dataset (Playtime data)
steam_dataset_path = kagglehub.dataset_download("nikdavis/steam-store-games")
print("Downloaded Steam Games Dataset (Playtime Data).")

# print("Contents:")
# print_directory_contents(steam_dataset_path)

Downloaded Steam Games Dataset (Playtime Data).


✅ Datasets successfully downloaded and ready for analysis:
- IGN Games Dataset: Professional critics' ratings
- Steam Games Dataset: User engagement metrics

## Data Loading and Exploration

Now let's load the datasets and examine their structure.

In [4]:
# Import data analysis libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

#### Load IGN Games Dataset

In [5]:
def load_ign_dataset(dataset_path):
    """Load and return the IGN games dataset with critics' ratings"""
    # Find CSV files in the dataset directory
    csv_files = list(Path(dataset_path).glob("ign.csv"))
    
    if csv_files:
        ign_df = pd.read_csv(csv_files[0])
        print(f"\033[1mIGN Dataset Shape\033[0m: {ign_df.shape}")
        print(f"\033[1mIGN Dataset Columns\033[0m: {list(ign_df.columns)}")
        print(f"\033[1mSample game titles\033[0m: {ign_df.iloc[:3, 2].values if len(ign_df) > 0 else 'No data'}")
        return ign_df
    else:
        print("❌ No CSV files found in IGN dataset directory")
        return None
    
# Load IGN dataset
ign_df = load_ign_dataset(ign_dataset_path)

IGN Dataset Shape: (18625, 11)
IGN Dataset Columns: ['Unnamed: 0', 'score_phrase', 'title', 'url', 'platform', 'score', 'genre', 'editors_choice', 'release_year', 'release_month', 'release_day']
Sample game titles: ['LittleBigPlanet PS Vita'
 'LittleBigPlanet PS Vita -- Marvel Super Hero Edition'
 'Splice: Tree of Life']


#### Load Steam Games Dataset

In [7]:
def load_steam_dataset(dataset_path):
    """Load and return the Steam games dataset with playtime statistics"""
    # Find CSV files in the dataset directory
    csv_files = list(Path(dataset_path).glob("steam.csv"))
    
    if csv_files:
        steam_df = pd.read_csv(csv_files[0])
        print(f"\033[1mSteam Dataset Shape\033[0m: {steam_df.shape}")
        print(f"\033[1mSteam Dataset Columns\033[0m: {list(steam_df.columns)}")
        print(f"\033[1mSample game titles\033[0m: {steam_df.iloc[:3, 1].values if len(steam_df) > 0 else 'No data'}")
        return steam_df
    else:
        print("❌ No CSV files found in Steam dataset directory")
        return None

# Load Steam dataset
steam_df = load_steam_dataset(steam_dataset_path)

Steam Dataset Shape: (27075, 18)
Steam Dataset Columns: ['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners', 'price']
Sample game titles: ['Counter-Strike' 'Team Fortress Classic' 'Day of Defeat']


#### Examine IGN Games Dataset

In [10]:
if ign_df is not None:
    print("IGN Dataset Preview (Critics' Ratings):")
    print("=" * 60)
    display(ign_df.head())
    
    print(f"\nIGN Dataset Info:")
    print("=" * 40)
    print(f"Rating columns: {[col for col in ign_df.columns if 'rating' in col.lower() or 'score' in col.lower()]}")
    ign_df.info()

IGN Dataset Preview (Critics' Ratings):


,Unnamed: 0,score_phrase,title,url,platform,score,genre,editors_choice,release_year,release_month,release_day
0,0,Amazing,LittleBigPlanet PS Vita,/games/littlebigplanet-vita/vita-98907,PlayStation Vita,9.0,Platformer,Y,2012,9,12
1,1,Amazing,LittleBigPlanet PS Vita -- Marvel Super Hero E...,/games/littlebigplanet-ps-vita-marvel-super-he...,PlayStation Vita,9.0,Platformer,Y,2012,9,12
2,2,Great,Splice: Tree of Life,/games/splice/ipad-141070,iPad,8.5,Puzzle,N,2012,9,12
3,3,Great,NHL 13,/games/nhl-13/xbox-360-128182,Xbox 360,8.5,Sports,N,2012,9,11
4,4,Great,NHL 13,/games/nhl-13/ps3-128181,PlayStation 3,8.5,Sports,N,2012,9,11



IGN Dataset Info:
Rating columns: ['score_phrase', 'score']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18625 entries, 0 to 18624
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      18625 non-null  int64  
 1   score_phrase    18625 non-null  object 
 2   title           18625 non-null  object 
 3   url             18625 non-null  object 
 4   platform        18625 non-null  object 
 5   score           18625 non-null  float64
 6   genre           18589 non-null  object 
 7   editors_choice  18625 non-null  object 
 8   release_year    18625 non-null  int64  
 9   release_month   18625 non-null  int64  
 10  release_day     18625 non-null  int64  
dtypes: float64(1), int64(4), object(6)
memory usage: 1.6+ MB


#### Examine Steam Games Dataset

In [11]:
if steam_df is not None:
    print("\n\nSteam Dataset Preview (Playtime Data):")
    print("=" * 60)
    display(steam_df.head())
    
    print(f"\nSteam Dataset Info:")
    print("=" * 40)
    print(f"Playtime columns: {[col for col in steam_df.columns if 'playtime' in col.lower() or 'time' in col.lower() or 'hour' in col.lower()]}")
    steam_df.info()



Steam Dataset Preview (Playtime Data):


,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,1273,267,258,184,5000000-10000000,3.99
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,FPS;Action;Sci-fi,0,5250,288,624,415,5000000-10000000,3.99



Steam Dataset Info:
Playtime columns: ['average_playtime', 'median_playtime']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27075 entries, 0 to 27074
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   appid             27075 non-null  int64  
 1   name              27075 non-null  object 
 2   release_date      27075 non-null  object 
 3   english           27075 non-null  int64  
 4   developer         27074 non-null  object 
 5   publisher         27061 non-null  object 
 6   platforms         27075 non-null  object 
 7   required_age      27075 non-null  int64  
 8   categories        27075 non-null  object 
 9   genres            27075 non-null  object 
 10  steamspy_tags     27075 non-null  object 
 11  achievements      27075 non-null  int64  
 12  positive_ratings  27075 non-null  int64  
 13  negative_ratings  27075 non-null  int64  
 14  average_playtime  27075 non-null  int64  
 15  median_p

## Data Cleaning and Preparation

Now we will clean and prepare the datasets for merging. This includes handling missing values, standardizing formats, and selecting relevant columns.

#### Clean IGN Dataframe

Cleaning steps for IGN Dataset:
1. Keep only relevant platforms (PC, Mac, Linux)
2. Convert release dates to datetime format for better analysis
3. Select key columns: title, platform, genre, release date, score, score_phrase
4. Remove games reviewed before 2003 (Steam's launch year)

In [27]:
# Filter and standardize platforms
ign_df_cleaned = ign_df.copy()

# Filter for PC-related platforms
ign_df_cleaned = ign_df_cleaned[ign_df_cleaned['platform'].isin(['PC'])]

# Convert score_phrase to categorical
ign_df_cleaned['score_phrase'] = pd.Categorical(ign_df_cleaned['score_phrase'])

# Convert genre to categorical
ign_df_cleaned['genre'] = pd.Categorical(ign_df_cleaned['genre'])

# Convert release date columns to datetime
ign_df_cleaned['release_date'] = pd.to_datetime({
    'year': ign_df_cleaned['release_year'],
    'month': ign_df_cleaned['release_month'],
    'day': ign_df_cleaned['release_day']
})

# Filter games after Steam's launch (2003)
ign_df_cleaned = ign_df_cleaned[ign_df_cleaned['release_year'] >= 2003]

# Select relevant columns
ign_df_cleaned = ign_df_cleaned[[
    'title', 'genre', 'release_date', 'score', 'score_phrase'
]]

# Reset index for clean representation
ign_df_cleaned = ign_df_cleaned.reset_index(drop=True)

print(f"Shape after cleaning: {ign_df_cleaned.shape}")
display(ign_df_cleaned.head())

Shape after cleaning: (2332, 5)


,title,genre,release_date,score,score_phrase
0,Guild Wars 2,RPG,2012-09-11,9.0,Amazing
1,Total War Battles: Shogun,Strategy,2012-09-11,7.0,Good
2,Mark of the Ninja,"Action, Adventure",2012-09-07,9.0,Amazing
3,Home: A Unique Horror Adventure,Adventure,2012-09-06,6.5,Okay
4,Mass Effect 3: Leviathan,RPG,2012-08-31,7.5,Good


#### Clean Steam Games Dataframe

Cleaning steps for Steam Dataset:
1. Convert release dates to datetime format
2. Calculate owners from range values
3. Keep only relevant platforms (PC, Mac, Linux)
4. Clean and categorize genres
5. Extract key metrics: title, platform, genre, release date, ratings, playtime, and owners

In [53]:
# Clean Steam Games DataFrame
steam_df_cleaned = steam_df.copy()

# Convert release_date to datetime
steam_df_cleaned['release_date'] = pd.to_datetime(steam_df_cleaned['release_date'])

# Calculate owners as midpoint of range
def extract_owners_midpoint(owners_range):
    low, high = map(lambda x: int(x.replace(',', '')), owners_range.split('-'))
    return (low + high) / 2

steam_df_cleaned['owners_count'] = steam_df_cleaned['owners'].apply(extract_owners_midpoint).astype(int)

# Filter for windows platform only
steam_df_cleaned = steam_df_cleaned[steam_df_cleaned['platforms'].str.contains('windows', case=False)]

# Convert genres to categorical
steam_df_cleaned['genres'] = pd.Categorical(steam_df_cleaned['genres'])
# Rename genres to genre for consistency with IGN dataset
steam_df_cleaned = steam_df_cleaned.rename(columns={'genres': 'genre'})

# Select relevant columns
steam_df_cleaned = steam_df_cleaned[[
    'name', 'genre', 'release_date', 'developer', 'publisher', 
    'required_age', 'positive_ratings', 'negative_ratings',
    'average_playtime', 'median_playtime', 'owners_count', 'price'
]]

# Rename columns for consistency
steam_df_cleaned = steam_df_cleaned.rename(columns={'name': 'title'})

# Reset index
steam_df_cleaned = steam_df_cleaned.reset_index(drop=True)

print(f"Shape after cleaning: {steam_df_cleaned.shape}")
display(steam_df_cleaned.head())

Shape after cleaning: (27070, 12)


,title,genre,release_date,developer,publisher,required_age,positive_ratings,negative_ratings,average_playtime,median_playtime,owners_count,price
0,Counter-Strike,Action,2000-11-01,Valve,Valve,0,124534,3339,17612,317,15000000,7.19
1,Team Fortress Classic,Action,1999-04-01,Valve,Valve,0,3318,633,277,62,7500000,3.99
2,Day of Defeat,Action,2003-05-01,Valve,Valve,0,3416,398,187,34,7500000,3.99
3,Deathmatch Classic,Action,2001-06-01,Valve,Valve,0,1273,267,258,184,7500000,3.99
4,Half-Life: Opposing Force,Action,1999-11-01,Gearbox Software,Valve,0,5250,288,624,415,7500000,3.99


NOTE: The differentiation between PC, Mac, and Linux versions can't be achieved here due to steam counting them as the same game. I decided to only consider PC, Mac, and Linux platforms while dropping the platform column as it would not be useful for analysis.

## Data Merging

Now we will merge the cleaned datasets on the game title to combine critics' ratings with user engagement metrics. This requires some title standardization to ensure accurate matching.

#### Clean Game Titles for Merging

In [54]:
def clean_game_names(df, title_column='title', clean_title_column='clean_title'):
    """Clean game titles for better matching between datasets"""
    df = df.copy()
    df[clean_title_column] = df[title_column].str.lower().str.strip()
    # Remove common suffixes/prefixes that might differ between platforms
    df[clean_title_column] = df['clean_title'].str.replace(r'\s*:\s*.*$', '', regex=True)  # Remove subtitles
    df[clean_title_column] = df['clean_title'].str.replace(r'[^\w\s]', '', regex=True)     # Remove punctuation
    df[clean_title_column] = df['clean_title'].str.replace(r'\s+', ' ', regex=True)        # Normalize whitespace
    return df

In [55]:
ign_df_with_clean_title = clean_game_names(ign_df_cleaned)

# Display title and cleaned title from ign_df_with_clean_title
print("IGN Dataset - Original vs Cleaned Titles:")
print("=" * 60)
display(ign_df_with_clean_title[['title', 'clean_title']].head(10))

IGN Dataset - Original vs Cleaned Titles:


,title,clean_title
0,Guild Wars 2,guild wars 2
1,Total War Battles: Shogun,total war battles
2,Mark of the Ninja,mark of the ninja
3,Home: A Unique Horror Adventure,home
4,Mass Effect 3: Leviathan,mass effect 3
5,Dark Souls (Prepare to Die Edition),dark souls prepare to die edition
6,Symphony,symphony
7,Tom Clancy's Ghost Recon Phantoms,tom clancys ghost recon phantoms
8,Thirty Flights of Loving,thirty flights of loving
9,World of Warcraft: Mists of Pandaria,world of warcraft


In [56]:
steam_df_with_clean_title = clean_game_names(steam_df_cleaned)

# Display title and cleaned title from steam_df_with_clean_title
print("\nSteam Dataset - Original vs Cleaned Titles:")
print("=" * 60)
display(steam_df_with_clean_title[['title', 'clean_title']].head(10))


Steam Dataset - Original vs Cleaned Titles:


,title,clean_title
0,Counter-Strike,counterstrike
1,Team Fortress Classic,team fortress classic
2,Day of Defeat,day of defeat
3,Deathmatch Classic,deathmatch classic
4,Half-Life: Opposing Force,halflife
5,Ricochet,ricochet
6,Half-Life,halflife
7,Counter-Strike: Condition Zero,counterstrike
8,Half-Life: Blue Shift,halflife
9,Half-Life 2,halflife 2


#### Merge Datasets on Cleaned Titles

We will perform an inner join on the cleaned titles to ensure we only keep games present in both datasets.

In [63]:
# Merge datasets on clean titles
merged_df = pd.merge(
    ign_df_with_clean_title,
    steam_df_with_clean_title,
    on='clean_title',
    how='inner',
    suffixes=('_ign', '_steam')
)

# Select and reorder relevant columns
merged_games_df = merged_df[[
    'title_ign',
    'release_date_ign',
    'release_date_steam',
    'genre_ign',
    'developer',
    'publisher',
    'required_age',
    'score',
    'score_phrase',
    'positive_ratings',
    'negative_ratings',
    'average_playtime',
    'median_playtime',
    'owners_count',
    'price',
]].copy()

# Calculate release date difference in days
merged_games_df['release_date_diff'] = (merged_games_df['release_date_steam'] - merged_games_df['release_date_ign']).dt.days

# Remove games where release dates differ by more than 30 days
merged_games_df = merged_games_df[abs(merged_games_df['release_date_diff']) <= 30]

# Drop the release_date_steam and release_date_diff columns as they're no longer needed
merged_games_df = merged_games_df.drop(['release_date_steam', 'release_date_diff'], axis=1)

# Rename columns for clarity and consistency
merged_games_df = merged_games_df.rename(columns={
    'title_ign': 'title',
    'release_date_ign': 'release_date',
    'genre_ign': 'genre',
    'score': 'critic_score',
    'score_phrase': 'critic_score_phrase',
    'positive_ratings': 'user_positive',
    'negative_ratings': 'user_negative',
    'average_playtime': 'average_playtime',
    'median_playtime': 'median_playtime',
    'owners_count': 'owners',
    'price': 'price',
})

print(f"Matching Results:")
print(f"   IGN games: {len(ign_df_cleaned):,}")
print(f"   Steam games: {len(steam_df_cleaned):,}")
print(f"   Matched games: {len(merged_games_df):,}")
print(f"   Match rate: {len(merged_games_df)/min(len(ign_df_cleaned), len(steam_df_cleaned))*100:.1f}%")

display(merged_games_df.head())


Matching Results:
   IGN games: 2,332
   Steam games: 27,070
   Matched games: 613
   Match rate: 26.3%


,title,release_date,genre,developer,publisher,required_age,critic_score,critic_score_phrase,user_positive,user_negative,average_playtime,median_playtime,owners,price
2,Home: A Unique Horror Adventure,2012-09-06,Adventure,Benjamin Rivers Inc.,Benjamin Rivers Inc.,0,6.5,Okay,1069,518,21,21,350000,1.99
3,Symphony,2012-08-30,Shooter,Empty Clip Studios,Empty Clip Studios,0,7.0,Good,1159,209,321,321,350000,6.19
4,Thirty Flights of Loving,2012-08-29,Adventure,Blendo Games,Blendo Games,0,8.0,Great,934,616,0,0,150000,3.99
5,Worms Revolution,2012-10-02,Strategy,Team17 Digital Ltd,Team17 Digital Ltd,0,8.5,Great,3922,686,445,259,1500000,10.99
6,Shad'O,2012-09-28,Adventure,Okugi Studio,Okugi Sudio,0,7.0,Good,50,42,58,58,35000,3.99


## Save Merged Dataset

Finally, we will save the merged dataset to a CSV file for future analysis and visualization.

In [ ]:
# Save the merged dataset to CSV
output_filename = "output.csv"
merged_games_df.to_csv(output_filename, index=False)
print(f"✅ Merged dataset saved to {output_filename}")
print(f"   Total rows: {len(merged_games_df):,}")
print(f"   Total columns: {len(merged_games_df.columns)}")

✅ Merged dataset saved to ign_steam_merged_games.csv
   Total rows: 613
   Total columns: 14
